# Laboratorio 2 — Deep Learning y catch22

**CC3084 Data Science · Universidad del Valle de Guatemala · Semestre II 2026 · Sección 20**

Diego López · Nelson Escalante · Roberto Nájera

Ingreso de viajeros internacionales a Guatemala (2009 – junio 2026). Continuación del
Laboratorio 1: se reutilizan las mismas series y las mismas particiones de entrenamiento
y prueba.

## Preparación

Las celdas de este documento **leen los resultados** que produce el pipeline
(`python main.py all`) y los presentan. Ningún número se calcula aquí ni se escribe a
mano: todo proviene de `results/*.json`, que es la única fuente de verdad.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

RAIZ = Path.cwd()
RESULTS = RAIZ / "results"


def cargar(nombre, pendiente_de=None):
    """Lee un JSON de results/. Avisa en vez de reventar si aun no existe."""
    ruta = RESULTS / f"{nombre}.json"
    if not ruta.exists():
        quien = f" Lo genera {pendiente_de}." if pendiente_de else ""
        display(Markdown(f"> Todavia no existe `results/{nombre}.json`.{quien}"))
        return None
    with open(ruta, encoding="utf-8") as fh:
        return json.load(fh)


def figura(ruta_relativa, ancho=760):
    display(Image(filename=str(RAIZ / ruta_relativa), width=ancho))


def tabla(filas, columnas):
    display(pd.DataFrame(filas, columns=columnas))

---
# 1. Modelos LSTM

## 1.1 Series y particiones utilizadas

El enunciado pide trabajar dos de las series del laboratorio anterior, con los mismos
conjuntos de entrenamiento y prueba.

In [ ]:
split = cargar("split")
lstm = cargar("lstm")

if split and lstm:
    tabla(
        [[s["clave"], split["train"]["inicio"], split["train"]["fin"],
          split["train"]["n_meses"], split["test"]["inicio"], split["test"]["fin"],
          split["test"]["n_meses"]]
         for s in lstm["series"]],
        ["Serie", "Train desde", "Train hasta", "Meses train",
         "Test desde", "Test hasta", "Meses test"],
    )

Se eligieron la **serie total** y la **vía Aérea**.

La vía Marítima quedó fuera de forma deliberada: sus últimos doce meses de entrenamiento
son cero exacto por el cierre de fronteras, de modo que una red con ventana de doce meses
solo observa ceros y su predicción recursiva colapsa a cero. Se verificó con cinco
configuraciones distintas y ninguna evita ese comportamiento. La serie sí participa del
ejercicio 2.

## 1.2 Configuraciones y tuneo de parámetros

Dos configuraciones por serie, con arquitecturas deliberadamente distintas para que la
comparación sea informativa.

In [ ]:
if lstm:
    filas = []
    for s in lstm["series"]:
        for nombre, info in s["modelos"].items():
            p = info["parametros"]
            filas.append([s["clave"], nombre, p["ventana"], p["unidades"], p["capas"],
                          p["dropout"], p["epochs_usadas"], round(p["loss_final"], 4)])
    tabla(filas, ["Serie", "Config", "Ventana", "Unidades", "Capas", "Dropout",
                  "Epocas elegidas", "Perdida final"])

### Tuneo

El número de épocas se eligió validando contra los últimos doce meses del entrenamiento.
El conjunto de prueba no interviene en la selección: usarlo sería fuga de datos y las
métricas finales dejarían de ser honestas.

In [ ]:
if lstm:
    for s in lstm["series"]:
        for nombre, info in s["modelos"].items():
            t = info["parametros"]["tuneo"]
            if not t:
                continue
            display(Markdown(f"**{s['clave']} — {nombre}** "
                             f"(validacion {t['val_inicio']} a {t['val_fin']}, "
                             f"{t['n_muestras_interno']} muestras)"))
            tabla([[r["epochs"], round(r["rmse_val"], 4),
                    "elegido" if r["epochs"] == t["mejor"] else ""]
                   for r in t["resultados"]],
                  ["Epocas", "RMSE validacion", ""])

> **Limitación.** Los doce meses de validación (2020-04 a 2021-03) coinciden con el
> colapso pandémico, así que el número de épocas seleccionado está sesgado hacia predecir
> bien una caída. Es consecuencia de tener 147 observaciones con la pandemia al final del
> tramo de entrenamiento.

## 1.3 Predicción con el mejor modelo

In [ ]:
comp = cargar("comparison")

if comp and lstm:
    claves = [s["clave"] for s in lstm["series"]]
    for s in comp["series"]:
        if s["clave"] not in claves:
            continue
        display(Markdown(f"### {s['nombre']} — gana **{s['ganador']['modelo']}**"))
        filas = sorted(([m, round(r["mae"], 1), round(r["rmse"], 1)]
                        for m, r in s["modelos"].items()), key=lambda f: f[2])
        tabla(filas, ["Modelo", "MAE", "RMSE"])
        figura(s["fig_forecast"])

## 1.4 Comparación con los modelos del laboratorio anterior

_(Texto de Roberto.)_

---
# 2. Exploración de similitud con catch22

## 2.1 Qué es catch22 y por qué importa

_(Texto pendiente.)_

## 2.2 – 2.4 Extracción de características y matriz

Se extraen las 22 características de **las siete series** del laboratorio anterior sobre
el tramo de entrenamiento, y se construye la matriz donde cada fila es una serie y cada
columna una característica.

In [ ]:
c22 = cargar("catch22")

if c22:
    display(Markdown(f"Matriz de **{c22['n_series']} series x {c22['n_features']} "
                     f"caracteristicas**."))
    display(pd.DataFrame(c22["matriz"], index=c22["series"],
                         columns=c22["features"]).round(3))

### Estandarización

Las 22 características viven en escalas muy distintas, así que se estandarizan
(z-score) **por columna** antes de cualquier análisis comparativo. Sin esto, las
características de rango grande dominarían las distancias y el PCA.

In [ ]:
if c22:
    display(pd.DataFrame(c22["matriz_estandarizada"], index=c22["series"],
                         columns=c22["features"]).round(3))

## 2.5 Análisis de la matriz

_(Figuras y resultados de Nelson: PCA, clustering, heatmap, correlaciones y distancias.)_

In [ ]:
analisis = cargar("catch22_analysis", pendiente_de="el analisis multivariado (2.5)")

if analisis:
    for clave, ruta in analisis.get("figuras", {}).items():
        display(Markdown(f"**{clave}**"))
        figura(ruta)

> **Sobre la matriz de correlaciones.** Con siete series y veintidós características, la
> matriz de datos tiene rango 6: la matriz de correlaciones resulta singular y muchos
> coeficientes valen ±1 por construcción matemática, no por una relación real entre
> características. Conviene interpretarla con esa reserva.

## 2.6 – 2.13 Análisis e interpretación

_(Respuestas de Roberto.)_

## 2.14 Modelo LSTM con las características de catch22

_(Modelo y discusión de Roberto.)_

In [ ]:
modelo_c22 = cargar("catch22_modelo", pendiente_de="el modelo del inciso 2.14")

if modelo_c22:
    display(modelo_c22)